# 04 -- Data Cleaning
`final_ml_ready_dataset.parquet` (`03` v3 ka output: currency-fixed **aur duration-fixed**) par asal data cleaning.

**Yaad rahe (`03` v3 se):** income-statement / cash-flow ke columns (Revenues, NetIncomeLoss...) ab **"per-quarter rate"** hain. 10-Q rows par asli quarter ki value, 10-K rows par saal ka average quarter (saal / 4). Balance-sheet columns waise hi hain.

**Is version mein tie-breaker fix shamil hai** (Step 3) -- amendment
dedup ab guaranteed original filing rakhti hai, chahe `filed` date
amendment ke barabar hi kyun na ho.

## Working directory fix (zaroori)

In [1]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("..")

print("Working directory:", os.getcwd())
assert Path("data/processed").exists(), "data/processed nahi mila -- path check karo"

Working directory: d:\Developing_Projects\cross-industry-financial-analysis_Project


## Setup

In [2]:
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("data/processed")
INPUT_PATH = PROCESSED_DIR / "final_ml_ready_dataset.parquet"
OUTPUT_PATH = PROCESSED_DIR / "04_cleaned_dataset.parquet"

ID_COLS = ["adsh", "cik", "name", "sic", "form", "period", "fy", "fp", "filed", "source_quarter"]

NON_NEGATIVE_COLS = [
    "Assets", "AssetsCurrent", "Liabilities", "LiabilitiesCurrent",
    "CashAndCashEquivalentsAtCarryingValue", "InventoryNet", "LongTermDebtNoncurrent",
]

TRANSITION_FORMS = ["10-KT", "10-QT", "10-KT/A", "10-QT/A"]
TIMELY_FILING_THRESHOLD_DAYS = 120

## Load raw (Stage 3) dataset

In [3]:
df = pd.read_parquet(INPUT_PATH)
print(f"Original shape: {df.shape}")
df.head()

Original shape: (59495, 32)


,adsh,cik,name,sic,form,period,fy,fp,filed,source_quarter,...,NetCashProvidedByUsedInFinancingActivities,NetCashProvidedByUsedInInvestingActivities,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,OperatingIncomeLoss,ResearchAndDevelopmentExpense,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues,SellingGeneralAndAdministrativeExpense,StockholdersEquity
0,0000002178-24-000035,2178,"ADAMS RESOURCES & ENERGY, INC.",5172,10-K,2023-12-31,2023,FY,2024-03-13,2024q1,...,-3.246000e+06,-7.780000e+05,7568750.0,53000.0,599750.0,NaN,NaN,6.863233e+08,NaN,9.271600e+07
1,0000002488-24-000012,2488,ADVANCED MICRO DEVICES INC,3674,10-K,2023-12-31,2023,FY,2024-01-31,2024q1,...,-2.865000e+08,-3.557500e+08,416750016.0,213500000.0,100250000.0,1.468000e+09,5.670000e+09,NaN,588000000.0,5.589200e+10
2,0000002969-24-000010,2969,"AIR PRODUCTS & CHEMICALS, INC.",2810,10-Q,2023-12-31,2024,Q1,2024-02-05,2024q1,...,1.362800e+09,-1.665600e+09,626600000.0,609299968.0,666899968.0,2.570000e+07,NaN,2.997400e+09,238400000.0,1.487300e+10
3,0000003499-24-000007,3499,ALEXANDERS INC,6798,10-K,2023-12-31,2023,FY,2024-02-12,2024q1,...,-2.310600e+07,8.045300e+07,27277750.0,25603250.0,NaN,NaN,NaN,5.624050e+07,NaN,2.376570e+08
4,0000003545-24-000034,3545,"ALICO, INC.",0100,10-Q,2023-12-31,2024,Q1,2024-02-07,2024q1,...,-4.448600e+07,7.522500e+07,-13169000.0,42945000.0,-17527000.0,NaN,1.398500e+07,NaN,NaN,2.877490e+08


In [4]:
metric_cols = [c for c in df.columns if c not in ID_COLS]
print(f"Metric columns ({len(metric_cols)})")

Metric columns (22)


## Step 1 -- Exact duplicate rows

In [5]:
before = len(df)
df = df.drop_duplicates()
print(f"Exact duplicates hataye -> {before - len(df)} rows removed")

Exact duplicates hataye -> 0 rows removed


## Step 2 -- Duplicate `adsh`

In [6]:
before = len(df)
df = df.drop_duplicates(subset=["adsh"], keep="first")
print(f"Duplicate 'adsh' hataye -> {before - len(df)} rows removed")

Duplicate 'adsh' hataye -> 0 rows removed


## Step 3 -- Duplicate `(cik, period)` -- amendments

**FIXED APPROACH + TIE-BREAKER:** `keep="first"` (ORIGINAL filing).

**Zaroori Tie-Breaker fix:** Kabhi original aur amendment ka `filed`
date EXACTLY SAME hota hai (jaise dono ek hi din file hui). Sirf
`filed` se sort karna is case mein original ko guarantee nahi rakhta --
ye row-order par depend kar sakta hai (coincidence). Isliye ek
`is_amendment` column banaya (form '/A' se khatam hoti hai ya nahi),
aur usay bhi sort mein shamil kiya -- taake original hamesha pehle
aaye, chahe filed date tie ho ya na ho.

In [7]:
df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
df["period"] = pd.to_datetime(df["period"], errors="coerce")
n_dup_before = df.duplicated(subset=["cik", "period"], keep=False).sum()
print(f"Duplicate (cik, period) combinations: {n_dup_before} rows")

example_ciks = df[df.duplicated(subset=["cik","period"], keep=False)]["cik"].unique()[:2]
df[df["cik"].isin(example_ciks)].sort_values(["cik","period"])[["cik","name","form","period","filed"]]

Duplicate (cik, period) combinations: 2384 rows


,cik,name,form,period,filed
291,314489,FIRST BUSEY CORP /NV/,10-K,2023-12-31,2024-02-23
292,314489,FIRST BUSEY CORP /NV/,10-K/A,2023-12-31,2024-02-23
5670,314489,FIRST BUSEY CORP /NV/,10-Q,2024-03-31,2024-05-07
12525,314489,FIRST BUSEY CORP /NV/,10-Q,2024-06-30,2024-08-06
18640,314489,FIRST BUSEY CORP /NV/,10-Q,2024-09-30,2024-11-05
24521,314489,FIRST BUSEY CORP /NV/,10-K,2024-12-31,2025-02-27
30034,314489,FIRST BUSEY CORP /NV/,10-Q,2025-03-31,2025-05-09
36293,314489,FIRST BUSEY CORP /NV/,10-Q,2025-06-30,2025-08-05
42205,314489,FIRST BUSEY CORP /NV/,10-Q,2025-09-30,2025-11-06
47933,314489,FIRST BUSEY CORP /NV/,10-K,2025-12-31,2026-02-26


In [8]:
# Tie-breaker column: True agar amendment hai (form '/A' se khatam)
df["is_amendment"] = df["form"].str.endswith("/A", na=False)

# Check: kitne cases mein filed date TIE hoti hai original aur amendment ke beech?
tied = (
    df[df.duplicated(subset=["cik","period"], keep=False)]
    .groupby(["cik","period"])["filed"]
    .nunique()
)
n_tied_cases = (tied == 1).sum()
print(f"Cases jaha filed date EXACTLY tie hoti hai (dono same din): {n_tied_cases}")

Cases jaha filed date EXACTLY tie hoti hai (dono same din): 24


In [9]:
before = len(df)
df = (
    df.sort_values(["filed", "is_amendment"])
    .drop_duplicates(subset=["cik", "period"], keep="first")
)
print(f"Purani/duplicate filings hatayi (sirf ORIGINAL rakhi, tie-breaker ke saath) -> {before - len(df)} rows removed")
print(f"Shape now: {df.shape}")

Purani/duplicate filings hatayi (sirf ORIGINAL rakhi, tie-breaker ke saath) -> 1254 rows removed
Shape now: (58241, 33)


## Step 4 -- Transition reports: FLAG, delete nahi
10-KT/10-QT -- fiscal year-end change, irregular period length.

In [10]:
df["is_transition_report"] = df["form"].isin(TRANSITION_FORMS)
n_transition = df["is_transition_report"].sum()
print(f"{n_transition} transition-report rows FLAG ki")

39 transition-report rows FLAG ki


## Step 5 -- Delinquent/late filer detection

In [11]:
df["filing_delay_days"] = (df["filed"] - df["period"]).dt.days
df["is_timely_filer"] = df["filing_delay_days"].between(0, TIMELY_FILING_THRESHOLD_DAYS)

print(f"Filing delay: median={df['filing_delay_days'].median():.0f} din, max={df['filing_delay_days'].max():.0f} din")
n_untimely = (~df["is_timely_filer"]).sum()
print(f"[FLAG] {n_untimely} rows 'is_timely_filer=False'")

df[~df["is_timely_filer"]][["cik","name","form","period","filed","filing_delay_days"]].sort_values("filing_delay_days", ascending=False).head()

Filing delay: median=44 din, max=2659 din
[FLAG] 2283 rows 'is_timely_filer=False'


,cik,name,form,period,filed,filing_delay_days
3145,1468978,"SOUTHERN ITS INTERNATIONAL, INC.",10-Q,2016-09-30,2024-01-11,2659
26568,1099234,"FRANKSPEECH NETWORK, INC.",10-Q,2018-12-31,2025-02-27,2250
39582,1372954,AMERITRUST CORP,10-Q/A,2020-12-31,2025-09-17,1721
20836,1397795,"BORN, INC.",10-K/A,2020-04-30,2024-12-19,1694
39583,1372954,AMERITRUST CORP,10-Q/A,2021-03-31,2025-09-17,1631


## Step 6 -- Identity-critical missing rows

In [12]:
before = len(df)
df = df.dropna(subset=["cik", "name", "adsh"])
print(f"Missing identity hataye -> {before - len(df)} rows removed")

Missing identity hataye -> 0 rows removed


## Step 7 -- Rows jinme SAARI financial metrics missing hain

In [13]:
before = len(df)
df = df.dropna(subset=metric_cols, how="all")
print(f"Fully-empty financial rows hataye -> {before - len(df)} rows removed")

Fully-empty financial rows hataye -> 0 rows removed


## Step 8 -- Logically-invalid values (sirf balance-sheet items)

In [14]:
for col in NON_NEGATIVE_COLS:
    if col in df.columns:
        invalid_mask = df[col] < 0
        n_invalid = invalid_mask.sum()
        if n_invalid > 0:
            print(f"[FLAG] {col}: {n_invalid} negative values -> NaN")
            df.loc[invalid_mask, col] = np.nan
        else:
            print(f"[OK] {col}: koi negative value nahi")

[OK] Assets: koi negative value nahi
[FLAG] AssetsCurrent: 2 negative values -> NaN
[OK] Liabilities: koi negative value nahi
[OK] LiabilitiesCurrent: koi negative value nahi
[OK] CashAndCashEquivalentsAtCarryingValue: koi negative value nahi
[OK] InventoryNet: koi negative value nahi
[OK] LongTermDebtNoncurrent: koi negative value nahi


## Step 9 -- dtype consistency + cleanup

In [15]:
for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# is_amendment ab zaroorat nahi -- sirf dedup step ke liye thi
df = df.drop(columns=["is_amendment"])
df[metric_cols].dtypes

Assets                                                 float32
AssetsCurrent                                          float32
CashAndCashEquivalentsAtCarryingValue                  float32
CostOfRevenue                                          float32
EarningsPerShareBasic                                  float32
EarningsPerShareDiluted                                float32
GrossProfit                                            float32
IncomeTaxExpenseBenefit                                float32
InventoryNet                                           float32
Liabilities                                            float32
LiabilitiesCurrent                                     float32
LongTermDebtNoncurrent                                 float32
NetCashProvidedByUsedInFinancingActivities             float32
NetCashProvidedByUsedInInvestingActivities             float32
NetCashProvidedByUsedInOperatingActivities             float32
NetIncomeLoss                                          

## Save cleaned dataset

In [16]:
df.to_parquet(OUTPUT_PATH, index=False)
print(f"[SAVED] {OUTPUT_PATH}")
print(f"Final cleaned shape: {df.shape}")

[SAVED] data\processed\04_cleaned_dataset.parquet
Final cleaned shape: (58241, 35)


## Summary: missingness after cleaning

In [17]:
missing_summary = df[metric_cols].isna().mean().sort_values(ascending=False) * 100
missing_summary.round(1)

CostOfRevenue                                          79.5
LongTermDebtNoncurrent                                 76.1
SellingGeneralAndAdministrativeExpense                 68.4
Revenues                                               68.1
ResearchAndDevelopmentExpense                          66.9
GrossProfit                                            61.7
InventoryNet                                           60.7
RevenueFromContractWithCustomerExcludingAssessedTax    60.4
IncomeTaxExpenseBenefit                                29.6
CashAndCashEquivalentsAtCarryingValue                  25.4
OperatingIncomeLoss                                    22.8
LiabilitiesCurrent                                     21.5
AssetsCurrent                                          21.2
EarningsPerShareDiluted                                18.5
EarningsPerShareBasic                                  16.4
NetCashProvidedByUsedInInvestingActivities             13.9
Liabilities                             

## Sanity checks

In [18]:
print(f"Duplicate (cik, period) left: {df.duplicated(subset=['cik','period']).sum()}  (should be 0)")
print(f"Unique companies (cik): {df['cik'].nunique()}")
print(f"Timely filers: {df['is_timely_filer'].sum()} ({df['is_timely_filer'].mean():.1%})")
print(f"Transition reports flagged: {df['is_transition_report'].sum()}")

Duplicate (cik, period) left: 0  (should be 0)
Unique companies (cik): 7582
Timely filers: 55958 (96.1%)
Transition reports flagged: 39
